In [18]:
import sys
from os.path import abspath, join, dirname, basename
import numpy as np
from project.vibronic import vIO, VMK, model_op
from project.vibronic_hamiltonian import vibronic_hamiltonian

def get_model_from_json_file(path, order):
    """ x """

    model = vIO.load_model_from_JSON(path)

    A, N = vIO._extract_dimensions_from_dictionary(model)

    if False:  # if the model includes the ground state that you excited it from
        model = vIO.model_remove_ground_state(model)

    if False:  # divide all off-diagonal (electronic) components by 2 (only if necessary)
        for a, b in it.product(range(A), range(A)):
            if a == b:
                model[VMK.E][a, b] /= 2

    model[VMK.etdm].fill(complex(0.1))
    model[VMK.mtdm].fill(complex(0.1))

    # swap electron / vibrational dimensions
    vIO.prepare_model_for_cc_integration(model, order)

    return model

hamiltonian_order = 1
t_order = 1 # can only ever be 1 
taylor_order = 1
z_order = 1

# get model for instantiation
path = join(f'model_test_oz.json')
model = get_model_from_json_file(path, order=hamiltonian_order)
model_name = 'test_oz'

# instantiate the vibronic_hamiltonian object
vh_instance = vibronic_hamiltonian(
    model, model_name,
    hamiltonian_truncation_order=hamiltonian_order, cc_truncation_order=taylor_order,
    T_truncation_order=t_order, Z_truncation_order=z_order,
    calculate_population_flag=False,
)

A, N = 1, 2
derv = {'dT_1': np.zeros((N))}
R = {0: np.zeros((A,A))}

# comes from dT definition 
T_dict = {0: np.zeros(A, dtype=complex),
    1: np.zeros((A, N,))}

# retrieved from where its init 
H_dict ={(0, 0): np.zeros((A, A), dtype=complex),
    (0, 1): np.zeros((A, A, N), dtype=complex),
    (1, 0): np.zeros((A, A, N), dtype=complex)}

Z_dict = {0: np.zeros((A, A), dtype=complex)}

print(R[0].shape)
print(vh_instance.dT)

# einsum in zeroth dt calc assumes you are in one of the stripped dimensions
print(T_dict[1][0,:].shape)
print(R[0]-np.einsum('i,i -> ',T_dict[1][0,:],derv['dT_1']))
zeroth_dT = (vibronic_hamiltonian.zeroth_dT(vh_instance,derv, R, T_dict[1][0,:]))
print(zeroth_dT.shape)


# zeroth_dT(None, derv, R, T_dict)

12 03:36:51   [INFO] _check_truncation_info: 
The Hamiltonian provided contains at most 1 order terms
Z truncation level is :1
T truncation level is :1

12 03:36:51   [INFO] __init__: Electronic TDM:
(1, 1)
12 03:36:51   [INFO] __init__: Magnetic TDM:
(1, 1)
12 03:36:51   [INFO] _initialize_hamiltonian: parameter of the Hamiltonian are entered properly!
12 03:36:51   [INFO] _initialize_hamiltonian: zero point energy: 0.59174550 ev


(1, 1)
{0: array([0.+0.j]), 1: array([[0.+0.j, 0.+0.j, 0.+0.j]])}
(2,)
[[0.]]


ValueError: einstein sum subscripts string contains too many subscripts for operand 0

In [ ]:
np.loadtxt(, dtype = float, comments=’#’, delimiter=None, converters=None, skiprows=0, usecols=None, unpack=False, ndmin=0, encoding=’bytes’, max_rows=None, *, like= None)